# Project Title: Reddit Big Data Pipeline

### Submission Date: 24th March 2026

### Importing Necessary Libaries

In [1]:

# IMPORTS AND SETUP

# Standard libraries
import os
import sys
import time
import requests
import psutil
from datetime import datetime
import logging

# Data processing
import pandas as pd
import numpy as np

# Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, from_unixtime, hour, dayofweek, month, year
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

# Dask
import dask.dataframe as dd

### Loggin Setup

In [2]:
# SETUP LOGGING
print(" Setting up logging...")

# Create logs directory
os.makedirs("logs", exist_ok=True)

# Create log filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"logs/pipeline_{timestamp}.log"

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),  # Save to file
        logging.StreamHandler()  # Also print to console
    ]
)

# Create logger
logger = logging.getLogger("RedditPipeline")

# Log startup
logger.info("="*60)
logger.info("REDDIT PIPELINE STARTED")
logger.info(f"Timestamp: {timestamp}")
logger.info("="*60)

print(f"✅ Logging setup complete! Log file: {log_file}")

2026-03-24 20:04:38,372 - INFO - ============================================================
2026-03-24 20:04:38,376 - INFO - REDDIT PIPELINE STARTED
2026-03-24 20:04:38,376 - INFO - Timestamp: 20260324_200438
2026-03-24 20:04:38,379 - INFO - ============================================================


 Setting up logging...
✅ Logging setup complete! Log file: logs/pipeline_20260324_200438.log


### Error Handling Functions

In [3]:
# ERROR HANDLING FUNCTIONS

import time
from functools import wraps

def retry(max_attempts=3, delay=2):
    """Retry decorator for transient failures"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    logger.info(f"Attempt {attempt + 1}/{max_attempts} for {func.__name__}")
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts - 1:
                        logger.error(f"All {max_attempts} attempts failed for {func.__name__}: {e}")
                        raise  # Re-raise on last attempt
                    logger.warning(f"Attempt {attempt + 1} failed: {e}. Retrying in {delay}s...")
                    time.sleep(delay)
            return None
        return wrapper
    return decorator

class PipelineError(Exception):
    """Custom exception for pipeline errors"""
    pass

def check_empty_dataframe(df, name="dataframe"):
    """Check if dataframe is empty and handle appropriately"""
    count = df.count()
    logger.info(f"{name} row count: {count:,}")
    
    if count == 0:
        logger.error(f"CRITICAL: {name} is empty!")
        raise PipelineError(f"{name} is empty - cannot continue")
    
    return df

def safe_read_parquet(file_list, select_cols=None):
    """Safely read parquet files with error handling"""
    try:
        logger.info(f"Attempting to read {len(file_list)} parquet files...")
        
        if select_cols:
            df = spark.read.parquet(*file_list).select(*select_cols)
        else:
            df = spark.read.parquet(*file_list)
        
        logger.info(f"[SUCCESS] Successfully read {df.count():,} rows")
        return df
        
    except Exception as e:
        logger.error(f"Failed to read parquet files: {e}")
        
        # Try with ignoreCorruptFiles
        logger.info("Retrying with ignoreCorruptFiles=true...")
        spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
        
        try:
            if select_cols:
                df = spark.read.parquet(*file_list).select(*select_cols)
            else:
                df = spark.read.parquet(*file_list)
            
            logger.info(f"✅ Successfully read {df.count():,} rows (with corrupted files ignored)")
            return df
            
        except Exception as e2:
            logger.error(f"Second attempt also failed: {e2}")
            raise PipelineError("Could not read data files")

### Loading Dataset Through API

In [4]:
#Downloaing dataset through HuggingFace API
url = "https://huggingface.co/api/datasets/fddemarco/pushshift-reddit-comments/parquet/default/train"

response = requests.get(url)

files = response.json()

print("Example files:")
print(files[:6])  # preview first files

Example files:
['https://huggingface.co/api/datasets/fddemarco/pushshift-reddit-comments/parquet/default/train/0.parquet', 'https://huggingface.co/api/datasets/fddemarco/pushshift-reddit-comments/parquet/default/train/1.parquet', 'https://huggingface.co/api/datasets/fddemarco/pushshift-reddit-comments/parquet/default/train/2.parquet', 'https://huggingface.co/api/datasets/fddemarco/pushshift-reddit-comments/parquet/default/train/3.parquet', 'https://huggingface.co/api/datasets/fddemarco/pushshift-reddit-comments/parquet/default/train/4.parquet', 'https://huggingface.co/api/datasets/fddemarco/pushshift-reddit-comments/parquet/default/train/5.parquet']


In [5]:
#Making Directory to store the dataset 
os.makedirs("data", exist_ok=True)

### Spark Session

In [6]:
#Starting Spark Session
spark = SparkSession.builder \
    .appName("RedditBigDataPipeline") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")
print(f"Cores: {spark.sparkContext.defaultParallelism}")

Spark version: 3.5.0
Spark UI: http://host.docker.internal:4042
Cores: 8


In [7]:
# Ignore corrupted files
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

In [8]:
# # Download first 4 files (~9GB total)
for file_url in files[:5]:
    filename = os.path.join("data", file_url.split("/")[-1])
    if not os.path.exists(filename):  # skip if already downloaded
        print("Downloading:", filename)
        r = requests.get(file_url, stream=True)
        with open(filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
print("Download completed")
print("Files in data folder:", os.listdir("data"))


Downloading: data\0.parquet
Downloading: data\1.parquet
Downloading: data\2.parquet
Downloading: data\3.parquet
Downloading: data\4.parquet
Download completed
Files in data folder: ['0.parquet', '1.parquet', '2.parquet', '3.parquet', '4.parquet']


In [9]:
# List of parquet files to check
files = [
    "data/0.parquet",
    "data/1.parquet",
    "data/2.parquet",
    "data/3.parquet",
    "data/4.parquet"
]

# Loop through each file and try to read it with Spark
for f in files:
    try:
        # Try reading the parquet file and show 1 row
        spark.read.parquet(f).show(1)
        
        # If successful, print OK
        print(f, "OK")
    except Exception as e:
        # If there is any error, mark file as BROKEN
        print(f, "BROKEN")

+-----------+----------------+--------------------+------------+-------+-----+-------+---------+--------+
|created_utc|controversiality|                body|subreddit_id|     id|score| author|subreddit| link_id|
+-----------+----------------+--------------------+------------+-------+-----+-------+---------+--------+
| 1325376000|               0|isn ' t this agai...|    t5_2qh3l|c3ctzsj|    5|goishin|     news|t3_nxrpq|
+-----------+----------------+--------------------+------------+-------+-----+-------+---------+--------+
only showing top 1 row

data/0.parquet OK
+-----------+----------------+--------------------+------------+-------+-----+--------+---------+--------+
|created_utc|controversiality|                body|subreddit_id|     id|score|  author|subreddit| link_id|
+-----------+----------------+--------------------+------------+-------+-----+--------+---------+--------+
| 1328054400|               0|gary payton bruce...|    t5_2qo4s|c3mmn28|    1|VexeD334|      nba|t3_p3qt5|


In [10]:
# Print file sizes (in GB) for all files in a folder
folder = "data"

for file in os.listdir(folder):
    path = os.path.join(folder, file)
    size = os.path.getsize(path) / (1024**3)   # convert to GB
    print(f"{file}: {size:.2f} GB")

0.parquet: 2.14 GB
1.parquet: 2.13 GB
2.parquet: 2.37 GB
3.parquet: 2.51 GB
4.parquet: 2.70 GB


### Data Ingestion

In [11]:
# Load parquet files safely with error handling and log performance
logger.info("Starting data ingestion phase")

start_time = time.time()

# List of files
files = ["data/0.parquet",
         "data/1.parquet", 
         "data/2.parquet", 
         "data/3.parquet", 
         "data/4.parquet"]

try:
    # Use safe read function
    df = safe_read_parquet(files, select_cols=["subreddit", "score", "created_utc"])
    
    # Check if empty
    df = check_empty_dataframe(df, "main dataframe")
    
    logger.info(f"[SUCCESS] Successfully loaded {df.count():,} rows")

    # Basic Mterics
    row_count = df.count()
    spark_end = time.time()
    spark_load_time = spark_end - start_time
    
    # Store in a variable for later use
    spark_load_time_value = spark_load_time
    print(f"Loaded {row_count:,} rows")
    print(f"Spark load time: {spark_load_time:.2f} seconds")
    logger.info(f"STAGE METRICS - Initial rows: {row_count:,}")
    memory_gb = psutil.Process().memory_info().rss / (1024**3)
    logger.info(f"STAGE METRICS - Memory usage: {memory_gb:.2f} GB")
    
except PipelineError as e:
    logger.error(f"Pipeline cannot continue: {e}")
    # Create dummy data for testing 
    logger.warning("Creating sample data for testing")
    from pyspark.sql.types import StructType, StructField, StringType, LongType
    schema = StructType([
        StructField("subreddit", StringType(), True),
        StructField("score", LongType(), True),
        StructField("created_utc", LongType(), True)
    ])
    df = spark.createDataFrame([("AskReddit", 100, 1328054400)], schema)
    logger.warning(f"Using sample data with {df.count()} rows")

2026-03-24 20:13:55,802 - INFO - Starting data ingestion phase
2026-03-24 20:13:55,804 - INFO - Attempting to read 5 parquet files...
2026-03-24 20:13:59,023 - INFO - [SUCCESS] Successfully read 82,660,155 rows
2026-03-24 20:14:00,851 - INFO - main dataframe row count: 82,660,155
2026-03-24 20:14:02,569 - INFO - [SUCCESS] Successfully loaded 82,660,155 rows
2026-03-24 20:14:03,939 - INFO - STAGE METRICS - Initial rows: 82,660,155
2026-03-24 20:14:03,939 - INFO - STAGE METRICS - Memory usage: 0.17 GB


Loaded 82,660,155 rows
Spark load time: 8.14 seconds


### Data Validation

In [12]:
#Performing data cleaning ans validation

df = df.dropna(subset=["subreddit", "score"])
df = df.filter(col("score") > -1000)

filtered_df = df.filter(col("score") > 0)

#Logging basic metrics
validated_count = filtered_df.count()
logger.info(f"STAGE METRICS - After validation: {validated_count:,} rows")
logger.info(f"STAGE METRICS - Removed {row_count - validated_count:,} rows ({(row_count-validated_count)/row_count*100:.1f}%)")

print("Validation complete")

2026-03-24 20:14:12,953 - INFO - STAGE METRICS - After validation: 74,962,882 rows
2026-03-24 20:14:12,954 - INFO - STAGE METRICS - Removed 7,697,273 rows (9.3%)


Validation complete


### Data Transformation

In [13]:
# Convert timestamp
df = df.withColumn("date", from_unixtime(col("created_utc")))

# Filter positive scores
filtered_df = df.filter(col("score") > 0)

filtered_df.show(10)

+---------------+-----+-----------+-------------------+
|      subreddit|score|created_utc|               date|
+---------------+-----+-----------+-------------------+
|           news|    5| 1325376000|2012-01-01 01:00:00|
|       politics|    5| 1325376000|2012-01-01 01:00:00|
|         gaming|    2| 1325376000|2012-01-01 01:00:00|
|       politics|    3| 1325376000|2012-01-01 01:00:00|
|            WTF|    1| 1325376001|2012-01-01 01:00:01|
|           pics|    2| 1325376001|2012-01-01 01:00:01|
|    harrypotter|    3| 1325376001|2012-01-01 01:00:01|
|leagueoflegends|    4| 1325376002|2012-01-01 01:00:02|
|leagueoflegends|    1| 1325376002|2012-01-01 01:00:02|
|  relationships|    2| 1325376002|2012-01-01 01:00:02|
+---------------+-----+-----------+-------------------+
only showing top 10 rows



In [14]:
# AGGREGATION - Top Subreddits
top_subreddits = filtered_df.groupBy("subreddit") \
    .agg(count("*").alias("comment_count")) \
    .orderBy(col("comment_count").desc())

# PERFORMANCE MONITORING #3 - AFTER AGGREGATION
subreddit_count = top_subreddits.count()
logger.info(f"STAGE METRICS - Unique subreddits: {subreddit_count:,}")

top_subreddits.show(5)

2026-03-24 20:14:37,650 - INFO - STAGE METRICS - Unique subreddits: 29,251


+---------+-------------+
|subreddit|comment_count|
+---------+-------------+
|AskReddit|     10114987|
|    funny|      3812375|
|     pics|      3028231|
|   gaming|      2326992|
|  atheism|      2095445|
+---------+-------------+
only showing top 5 rows



In [15]:
# Window Function
avg_scores = df.groupBy("subreddit") \
    .agg(avg("score").alias("avg_score"))

window_spec = Window.orderBy(col("avg_score").desc())

ranked_subreddits = avg_scores.withColumn(
    "rank",
    rank().over(window_spec)
)

ranked_subreddits.show(10)

+-------------------+------------------+----+
|          subreddit|         avg_score|rank|
+-------------------+------------------+----+
|             cognac|132.07692307692307|   1|
|         Bestof2011| 25.47137518684604|   2|
|  tvcharacterbattle|22.190594059405942|   3|
|forwardsfromgrandma|21.428571428571427|   4|
|      AlienBlueNews|              17.0|   5|
|            TeenR4R|              16.0|   6|
|             t:zero|15.910828025477707|   7|
| Shitty_Watercolour|15.137873754152825|   8|
|            Scatsex|              15.0|   9|
|        AlexisTexas|              15.0|   9|
+-------------------+------------------+----+
only showing top 10 rows



## Data Storage in Parquet Format

In [16]:
# Environment configuration for local installations
import os

# Set JAVA_HOME for local installations (macOS/Linux)
# This is only needed for local development - Colab has Java pre-installed
java_paths = [
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",  # macOS Apple Silicon
    "/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",     # macOS Intel
    "/usr/lib/jvm/java-17-openjdk-amd64",                              # Ubuntu/Debian
]

java_home = None

for path in java_paths:
    if os.path.exists(path):
        os.environ["JAVA_HOME"] = path
        java_home = path
        break

if java_home:
    print(f"✓ JAVA_HOME set to: {java_home}")
else:
    print("⚠️ Java not found. PySpark requires Java 17.")
    print("  On macOS: brew install openjdk@17")
    print("  On Ubuntu: sudo apt install openjdk-17-jdk")

✓ JAVA_HOME set to: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home


In [17]:
#Storage (Parquet)
top_subreddits.write.mode("overwrite").parquet("output/top_subreddits")

ranked_subreddits.write.mode("overwrite").parquet("output/ranked_subreddits")

print("Results stored successfully")

26/03/24 19:46:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 19:46:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 19:46:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 19:46:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 19:46:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 19:46:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/24 1

Results stored successfully


In [16]:
# END OF PIPELINE
end_time = time.time()

# PERFORMANCE MONITORING #4 - AT THE END
total_time = end_time - start_time
logger.info(f"STAGE METRICS - Total pipeline time: {total_time:.2f} seconds")

# If you have peak_memory variable, 
if 'peak_memory' in locals():
    logger.info(f"STAGE METRICS - Peak memory: {peak_memory:.2f} GB")

print("Pipeline execution time:", end_time - start_time)

2026-03-24 20:15:21,348 - INFO - STAGE METRICS - Total pipeline time: 85.54 seconds


Pipeline execution time: 85.54395294189453


### Comparison Spark Vs Dask

In [17]:
logger.info("="*60)
logger.info("STARTING DASK VS SPARK COMPARISON")
logger.info("="*60)

comparison_results = []

2026-03-24 20:15:26,377 - INFO - ============================================================
2026-03-24 20:15:26,378 - INFO - STARTING DASK VS SPARK COMPARISON
2026-03-24 20:15:26,379 - INFO - ============================================================


In [18]:
# TEST 1: DATA LOADING 
logger.info("\n TEST 1: Data Loading Time")

# Spark load time (approximate from earlier)
spark_load_time = spark_load_time_value  

# Dask load time
start = time.time()
ddf = dd.read_parquet(["data/0.parquet", "data/1.parquet", "data/2.parquet", "data/3.parquet", "data/4.parquet"])
dask_load_time = time.time() - start
logger.info(f"  Dask load time: {dask_load_time:.2f}s")

comparison_results.append({
    'Operation': 'Load Data',
    'Spark (s)': round(spark_load_time, 2),
    'Dask (s)': round(dask_load_time, 2),
    'Winner': 'Dask' if dask_load_time < spark_load_time else 'Spark'
})

2026-03-24 20:15:28,847 - INFO - 
 TEST 1: Data Loading Time
2026-03-24 20:15:29,011 - INFO -   Dask load time: 0.16s


In [19]:
# TEST 2: FILTER OPERATION 
logger.info("\n TEST 2: Filter (score > 0)")

# Spark
start = time.time()
spark_filtered_count = df.filter(col("score") > 0).count()
spark_filter_time = time.time() - start

# Dask
start = time.time()
dask_filtered = ddf[ddf["score"] > 0]
dask_filtered_count = len(dask_filtered)
dask_filter_time = time.time() - start

logger.info(f"  Spark: {spark_filter_time:.2f}s, Dask: {dask_filter_time:.2f}s")
logger.info(f"  Spark result count: {spark_filtered_count:,}")
logger.info(f"  Dask result count: {dask_filtered_count:,}")

comparison_results.append({
    'Operation': 'Filter',
    'Spark (s)': round(spark_filter_time, 2),
    'Dask (s)': round(dask_filter_time, 2),
    'Winner': 'Dask' if dask_filter_time < spark_filter_time else 'Spark'
})

2026-03-24 20:15:34,002 - INFO - 
 TEST 2: Filter (score > 0)
2026-03-24 20:15:41,204 - INFO -   Spark: 2.72s, Dask: 4.48s
2026-03-24 20:15:41,206 - INFO -   Spark result count: 74,962,882
2026-03-24 20:15:41,207 - INFO -   Dask result count: 74,962,882


In [20]:
#TEST 3: GROUPBY AGGREGATION 
logger.info("\n TEST 3: GroupBy Aggregation")

# Spark
start = time.time()
spark_agg = df.groupBy("subreddit").count().collect()
spark_agg_time = time.time() - start

# Dask
start = time.time()
dask_agg = ddf.groupby("subreddit").size().compute()
dask_agg_time = time.time() - start
dask_result_count = len(dask_agg)

logger.info(f"  Spark: {spark_agg_time:.2f}s, Dask: {dask_agg_time:.2f}s")
logger.info(f"  Unique subreddits found: {dask_result_count:,}")

comparison_results.append({
    'Operation': 'GroupBy',
    'Spark (s)': round(spark_agg_time, 2),
    'Dask (s)': round(dask_agg_time, 2),
    'Winner': 'Dask' if dask_agg_time < spark_agg_time else 'Spark'
})

2026-03-24 20:17:15,690 - INFO - 
 TEST 3: GroupBy Aggregation
2026-03-24 20:17:32,607 - INFO -   Spark: 7.17s, Dask: 9.74s
2026-03-24 20:17:32,609 - INFO -   Unique subreddits found: 29,522


In [21]:
# TEST 4: MEMORY USAGE
process = psutil.Process()
memory_gb = process.memory_info().rss / (1024**3)
logger.info(f"\n Memory Usage: {memory_gb:.2f} GB")

2026-03-24 20:18:16,388 - INFO - 
 Memory Usage: 0.10 GB


In [22]:
# COMPARISON TABLE 
logger.info("\n" + "="*60)
logger.info("PERFORMANCE COMPARISON RESULTS")
logger.info("="*60)

comparison_df = pd.DataFrame(comparison_results)
print("\n")
print(comparison_df.to_string(index=False))

2026-03-24 20:18:19,182 - INFO - 
2026-03-24 20:18:19,183 - INFO - PERFORMANCE COMPARISON RESULTS
2026-03-24 20:18:19,185 - INFO - ============================================================




Operation  Spark (s)  Dask (s) Winner
Load Data       8.14      0.16   Dask
   Filter       2.72      4.48  Spark
  GroupBy       7.17      9.74  Spark


In [23]:
spark.stop()